In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [10]:
df = pd.read_csv('../data/train_data.csv')
df['price_billion'] = df['price_total'] / 1000000000

In [11]:
q_low_price = df['price_billion'].quantile(0.01)
q_hi_price  = df['price_billion'].quantile(0.99)
q_low_area  = df['area'].quantile(0.01)
q_hi_area   = df['area'].quantile(0.99)

# Tạo df_clean chứa dữ liệu đã lọc sạch (Dùng .copy() để tránh cảnh báo của Pandas)
df_clean = df[(df['price_billion'] >= q_low_price) & (df['price_billion'] <= q_hi_price) &
              (df['area'] >= q_low_area) & (df['area'] <= q_hi_area)].copy()

print(f"Số dòng ban đầu: {len(df)} | Sau khi lọc Outlier: {len(df_clean)}")

Số dòng ban đầu: 32427 | Sau khi lọc Outlier: 31245


In [12]:
# 4.1. Gộp số phòng tránh đa cộng tuyến
df_clean['total_rooms'] = df_clean['num_bedrooms'] + df_clean['num_toilets']

# 4.2. Gộp điểm tiện ích xung quanh
df_clean['amenity_score'] = df_clean['num_schools_1km'] + df_clean['num_hospitals_2km'] + df_clean['num_markets_1km']

# 4.3. Tạo đặc trưng mới: Diện tích trung bình trên mỗi phòng (Cộng 1 để tránh lỗi chia cho 0)
df_clean['area_per_room'] = df_clean['area'] / (df_clean['total_rooms'] + 1)

In [13]:
y = df_clean['price_billion']

# Cực kỳ quan trọng: Xóa các cột gây rò rỉ dữ liệu (Leakage) VÀ các cột cũ đã được gộp ở Bước 4
cols_to_drop =[
    'price_total', 'price_per_m2', 'price_billion', # Cột giá (Leakage)
    'num_bedrooms', 'num_toilets',                  # Cột phòng cũ
    'num_schools_1km', 'num_hospitals_2km', 'num_markets_1km' # Cột tiện ích cũ
]
X = df_clean.drop(columns=cols_to_drop)

# Chuyển đổi các biến phân loại (Categorical) thành dạng số (One-Hot Encoding)
X = pd.get_dummies(X, columns=['category', 'district', 'province', 'legal_status'], drop_first=True)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Kích thước tập Train:", X_train.shape)

Kích thước tập Train: (24996, 88)


In [15]:
print("Đang huấn luyện mô hình XGBoost...")
# Biến đổi Logarit cho y_train để AI học tốt hơn
y_train_log = np.log1p(y_train)

# Khởi tạo mô hình XGBoost với các tham số chống Overfitting
model = XGBRegressor(
    n_estimators=600,        # Tăng số cây quyết định
    learning_rate=0.03,      # Học chậm lại để tăng độ chính xác
    max_depth=8,             # Độ sâu của cây
    subsample=0.8,           # Lấy mẫu ngẫu nhiên 80% data
    colsample_bytree=0.8,    # Lấy mẫu ngẫu nhiên 80% đặc trưng
    random_state=42,
    n_jobs=-1                # Dùng tối đa CPU
)

# Train model
model.fit(X_train, y_train_log)
print("Huấn luyện xong!")

Đang huấn luyện mô hình XGBoost...
Huấn luyện xong!


In [16]:
# Model dự đoán ra giá trị Logarit
y_pred_log = model.predict(X_test)

# Chuyển đổi ngược (Exponential) từ Logarit về giá trị tỷ VNĐ thực tế
y_pred = np.expm1(y_pred_log)

# In kết quả
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("-" * 30)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH:")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f} Tỷ VNĐ")
print("-" * 30)

------------------------------
KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH:
R2 Score: 0.8301
MAE: 2.9011 Tỷ VNĐ
------------------------------
